# Import Packages #

In [1]:
# 
import numpy as np
import pandas as pd
import time
import os
import math

#
import geopandas as gpd
from geopandas import datasets, GeoDataFrame, read_file
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.stattools import durbin_watson
from statsmodels.stats.diagnostic import het_breuschpagan
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
from sklearn.cluster import KMeans
from scipy.stats import shapiro
from scipy.spatial import distance

#
import plotly.graph_objects as go
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import matplotlib.lines as mlines
from matplotlib.pyplot import MultipleLocator
from matplotlib.ticker import AutoMinorLocator
from matplotlib.ticker import MaxNLocator
from matplotlib.colors import ListedColormap

# Generate Figure #

In [ ]:
#
Data = pd.read_csv('/Volumes/IMAC/Research/全国点评研究/Figures/Sub-figures/Mall2015-2023joincity287.csv')
# 
Data['NSF'] = Data['Count'] * Data['P_Horizontal']
Data['NGF'] = Data['Count'] * Data['P_Vertical']
# 
result = Data.groupby('Year').agg({'Count': 'sum', 'NGF': 'sum', 'NSF': 'sum'}).reset_index()
result['P_NGF'] = result['NGF'] / result['Count'] * 100  
result['P_NSF'] = result['NSF'] / result['Count'] * 100  
result['Count'] /= 1_000_000  
# 
colors = {2015: '#6eaed1', 2019: '#fed06e', 2023: '#dc6e57'}
# 
for col, y_label in zip(['Count'], ['Total count (millions)']):
    #
    plt.figure(figsize=(4, 4))
    # 
    bars = plt.bar(result['Year'], result[col], color=[colors.get(year, '#000000') for year in result['Year']], width=0.9)   
    # 
    for bar in bars:
        #
        plt.text(bar.get_x() + bar.get_width() / 2, bar.get_height(), f'{bar.get_height():.2f}', ha='center', va='bottom', fontsize=10)
    # 
    plt.xlabel('Year', fontsize=12)
    plt.ylabel(y_label, fontsize=10)
    plt.xticks(result['Year'], fontsize=10)
    plt.yticks(range(0, 21, 5), fontsize=10)
    # 
    plt.tight_layout()
    plt.savefig(f"/Volumes/IMAC/Research/Figures/Sub-figures/{col}_by_Year.eps", format="eps")
    plt.show()

In [ ]:
# 
Data['NSF'] = Data['Count'] * Data['P_Horizontal']
Data['NGF'] = Data['Count'] * Data['P_Vertical']
# 
result = Data.groupby('Year', as_index=False)[['Count', 'NGF', 'NSF']].sum()
result['P_NGF'] = result['NGF'] / result['Count'] * 100  # Convert to percentage
result['P_NSF'] = result['NSF'] / result['Count'] * 100  # Convert to percentage
result['Count'] = result['Count'] / 1_000_000          # Convert to millions
# 
colors = {2015: '#6eaed1', 2019: '#fed06e', 2023: '#dc6e57'}
# 
columns = ['Count', 'P_NGF', 'P_NSF']
y_labels = ['Total count (millions)', 'Percentage of non-ground-floor shops (%)', 'Percentage of non-street-facing shops (%)']
# 
colors = {2015: '#6eaed1', 2019: '#fed06e', 2023: '#dc6e57'}
# 
dual_axis_columns = [('NGF', 'P_NGF', 'Count of non-ground-floor shops (millions)', 'Percentage of non-ground-floor shops'),
                     ('NSF', 'P_NSF', 'Count of non-street-facing shops (millions)', 'Percentage of non-street-facing shops')]
#
for left_col, right_col, left_label, right_label in dual_axis_columns:
    #
    plt.figure(figsize=(4, 4))  
    #
    bar_width = 0.9  # 
    bars = plt.bar(result['Year'], result[left_col] / 1_000_000, color=[colors[year] for year in result['Year']], width=bar_width)
    # 
    for bar in bars:
        #
        height = bar.get_height()
        plt.text(bar.get_x() + bar.get_width() / 2, height, f'{height:.2f}', ha='center', va='bottom', fontsize=10)
    # 
    plt.xlabel('Year', fontsize=12)
    plt.ylabel(left_label, fontsize=10)
    plt.xticks(result['Year'], fontsize=10)
    plt.ylim(0, 10) 
    plt.yticks(fontsize=10)
    # 
    ax2 = plt.gca().twinx()  
    ax2.plot(result['Year'], result[right_col], color='red', marker='o', label=right_label)
    ax2.set_ylabel(right_label, fontsize=10, color='red')
    ax2.tick_params(axis='y', labelcolor='red', labelsize=10)
    ax2.set_ylim(0, 51)  # 
    # 
    offset = 50 * 0.03  # 
    #
    for x, y in zip(result['Year'], result[right_col]):
        #
        ax2.text(x, y - offset, f'{y:.2f}%', ha='center', va='top', fontsize=10, color='red')  
    # 
    plt.tight_layout()
    filename = f"/Volumes/IMAC/Research/Figures/Sub-figures/{right_col}_by_Year.eps"
    plt.savefig(filename, format="eps")
    plt.show()

In [ ]:
# 
Datatest = Data[['City', 'CityTier', 'Year', 'P_GF', 'P_FS10', 'P_FS15', 'P_FS20', 'P_FS25', 'P_FS30']]
# 
Datatest = Datatest.rename(columns={'P_FS10': 'P_NSF10','P_FS15': 'P_NSF15','P_FS20': 'P_NSF20','P_FS25': 'P_NSF25','P_FS30': 'P_NSF30'})
Datatest[['P_NSF10', 'P_NSF15', 'P_NSF20', 'P_NSF25', 'P_NSF30']] = (1 - Datatest[['P_NSF10', 'P_NSF15', 'P_NSF20', 'P_NSF25', 'P_NSF30']])*100
# 
colors = {2015: '#6eaed1', 2019: '#fed06e',2023: '#dc6e57'}
# 
data_melted = Datatest.melt(id_vars=['City', 'CityTier', 'Year'], value_vars=['P_NSF10', 'P_NSF15', 'P_NSF20', 'P_NSF25', 'P_NSF30'], 
                            var_name='column', value_name='value')
# 
g = sns.catplot(x='column', y='value', hue='Year', data=data_melted, kind='point', height=6, aspect=1.5, palette=list(colors.values()), capsize=0.2,
                legend=False)
# 
for year in [2015, 2019]:
    #
    data_year = Datatest[Datatest['Year'] == year][['P_NSF10', 'P_NSF15', 'P_NSF20', 'P_NSF25', 'P_NSF30']]
    means = data_year.median().values
    errors = data_year.sem().values * 1.96  # 95% confidence interval
    #
    for i, mean in enumerate(means):g.ax.errorbar(i, mean, yerr=errors[i], fmt='o', color=colors[year], linestyle='--', linewidth=0.001, markersize=0.5)
# 
plt.xticks(range(5), ['10m', '15m', '20m', '25m', '30m'])      
plt.xlabel('Results using different thresholds')
plt.ylabel('Percentage of non-street-facing shops (%)')
# 
plt.gca().spines['top'].set_visible(True)
plt.gca().spines['right'].set_visible(True)
# 
g.ax.legend(title='Year', loc='upper right')
plt.savefig(f'/Volumes/IMAC/Research/Figures/Sub-figures/RT1.eps', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# 
sns.set_theme(style="white")
sns.set_style('white', rc={'xtick.bottom': True,'ytick.left': True,})
# Define the color palette for the plot based on the years
colors = {2015: '#6eaed1', 2019:'#fed06e', 2023:'#dc6e57'}
columns_to_plot = ['P_Horizontal', 'P_Vertical']
#
LP = [0.033, 0.0715]
i = 0
# 
for column in columns_to_plot:
    # 
    filtered_data = Data[(Data[column] != 0) & (Data[column] != 1)]
    # 
    filtered_data = filtered_data.groupby('City').filter(lambda x: len(x) == 3)
    # 
    filtered_data[column] = filtered_data[column] * 100
    # 
    years = filtered_data['Year'].unique()
    # 
    fig, ax = plt.subplots(figsize=(8, 5), dpi=300)
    # 
    sns.kdeplot(data=filtered_data, x=column, hue='Year', fill=True, bw_adjust=0.15, palette=colors, hue_order=sorted(years))
    #
    for year in sorted(years):
        # 
        mean_value = filtered_data[filtered_data['Year'] == year][column].mean()
        # 
        color = colors[year]
        #
        plt.axvline(mean_value, color=color, linestyle='dashed', linewidth=1, label=f'Mean ({year})')
        # 
        label_position = LP[i]  # P
        PositionX = mean_value
        if year == 2019: PositionX = mean_value + 0.25
        if year == 2015: PositionX = mean_value - 0.25
        plt.text(PositionX, label_position, f'{mean_value:.2f}', color=color, ha='center', va='center', fontsize=10)
    #
    i += 1
    # 
    if column == 'P_Horizontal': plt.xlabel('Percentage of non-street-facing shops (%)', fontsize=12)
    elif column == 'P_Vertical': plt.xlabel('Percentage of non-ground-floor shops (%)', fontsize=12)
    #
    plt.ylabel('Density', fontsize=12)
    # 
    plt.legend(title='Year', loc='upper right', fontsize=10)
    # 
    plt.tick_params(axis='both', direction='out', length=6, width=1)  # 
    # 
    ax.minorticks_off()
    # 
    plt.savefig(f'/Volumes/IMAC/Research/Figures/Sub-figures/kde_plot_{column}.eps', dpi=300, bbox_inches='tight') 
    # 
    plt.show()

In [ ]:
# 
colors = {2015: '#6eaed1', 2019:'#fed06e', 2023:'#dc6e57'}
# 
tier_order = ['Tier 1', 'Tier 2', 'Tier 3', 'Tier 4', 'Tier 5']
columns_to_plot = ['P_Horizontal', 'P_Vertical']
#
sns.set_style('white', rc={'xtick.bottom': True,'ytick.left': True,})
# 
for i, column in enumerate(columns_to_plot):  
    #
    filtered_data = Data[(Data[column] != 0) & (Data[column] != 1)]
    # 
    filtered_data = filtered_data.groupby('City').filter(lambda x: len(x) == 3) 
    # 
    filtered_data[column] = filtered_data[column] * 100
    # 
    fig, ax = plt.subplots(figsize=(8, 5), dpi=300)
    # 
    years = filtered_data['Year'].unique()
    # 
    sns.violinplot(data=filtered_data, x='CityTier', y=column, hue='Year', inner="quart", fill=False, palette=colors)
    # 
    plt.xlabel('City tier')
    #
    if column == 'P_Horizontal': plt.ylabel('Percentage of non-street-facing shops (%)')
    elif column == 'P_Vertical': plt.ylabel('Percentage of non-ground-floor shops (%)')
    # 
    plt.ylim(0, 75)
    # 
    if i == 0: plt.legend(title='Year', loc='lower left')
    else: plt.legend(title='Year', loc='upper right')
    # 
    plt.tick_params(axis='both', direction='out', length=6, width=1) 
    # 
    ax.minorticks_off()
    # 
    plt.savefig(f'/Volumes/IMAC/Research/Figures/Sub-figures/violinplot_{column}.eps', dpi=300, bbox_inches='tight')
    # 
    plt.show()

In [ ]:
#
cities = Data['City'].unique()
# 
colors = {2015: '#6eaed1', 2019:'#fed06e', 2023:'#dc6e57'}
# 
tier_styles = {'Tier 1': {'marker': 'o', 'color': '#6eaed1'},'Tier 2': {'marker': 's', 'color': '#6eaed1'},'Tier 3': {'marker': '^','color': '#6eaed1'},
               'Tier 4': {'marker': 'D', 'color': '#6eaed1'},'Tier 5': {'marker': 'x', 'color': '#6eaed1'}}
#
legend_labels = {tier: patches.Patch(color=style['color'], label=tier) for tier, style in tier_styles.items()}
# 
annotated_points = []
# 
for tier, style in tier_styles.items():
    # 
    fig, ax = plt.subplots(figsize=(4, 4), dpi=300)
    # 
    cities_in_tier = Data[Data['CityTier'] == tier]['City'].unique()
    # 
    for city in cities_in_tier:
        #
        city_data = Data[Data['City'] == city]
        city_years = city_data['Year'].unique()
        city_data['P_Horizontal'] = city_data['P_Horizontal'] * 100
        city_data['P_Vertical'] = city_data['P_Vertical'] * 100
        # 
        if set([2015, 2019, 2023]).issubset(city_years):
            # 
            coords_2015 = city_data[city_data['Year'] == 2015].iloc[0]
            coords_2019 = city_data[city_data['Year'] == 2019].iloc[0]
            coords_2023 = city_data[city_data['Year'] == 2023].iloc[0]
            city_name = coords_2015['CityName_x']  # 使用 CityName_x 字段
            # 
            dx_1 = coords_2019['P_Horizontal'] - coords_2015['P_Horizontal']
            dy_1 = coords_2019['P_Vertical'] - coords_2015['P_Vertical']
            dx_2 = coords_2023['P_Horizontal'] - coords_2019['P_Horizontal']
            dy_2 = coords_2023['P_Vertical'] - coords_2019['P_Vertical']
            # 
            arrow1 = patches.FancyArrowPatch((coords_2015['P_Horizontal'], coords_2015['P_Vertical']),
                                             (coords_2015['P_Horizontal'] + dx_1, coords_2015['P_Vertical'] + dy_1),color='#fde3d6', arrowstyle='->', 
                                             lw=1.5)
            #
            ax.add_patch(arrow1)
            # 
            arrow2 = patches.FancyArrowPatch((coords_2019['P_Horizontal'], coords_2019['P_Vertical']),
                                             (coords_2019['P_Horizontal'] + dx_2, coords_2019['P_Vertical'] + dy_2),color='#fed06e', arrowstyle='->',
                                             lw=1.5)
            #
            ax.add_patch(arrow2)
            # 
            ax.plot(coords_2015['P_Horizontal'], coords_2015['P_Vertical'], marker=style['marker'], color=colors[2015])
            ax.plot(coords_2019['P_Horizontal'], coords_2019['P_Vertical'], marker=style['marker'], color=colors[2019])
            ax.plot(coords_2023['P_Horizontal'], coords_2023['P_Vertical'], marker=style['marker'], color=colors[2023])
            # 
            def is_dense_area(point, threshold=3):
                # 
                for annotated_point in annotated_points:
                    #
                    if distance.euclidean(point, annotated_point) < threshold:
                        #
                        return True
                #
                return False
            #
            point_2015 = (coords_2015['P_Horizontal'], coords_2015['P_Vertical'])
            #
            if not is_dense_area(point_2015):
                #
                ax.text(point_2015[0], point_2015[1], city_name, fontsize=4, ha='right')
                annotated_points.append(point_2015)
            #
            point_2023 = (coords_2023['P_Horizontal'], coords_2023['P_Vertical'])
            #
            if not is_dense_area(point_2023):
                #
                ax.text(point_2023[0], point_2023[1], city_name, fontsize=4, ha='right')
                annotated_points.append(point_2023)
    # 
    ax.yaxis.set_major_locator(MaxNLocator(integer=True))
    # 
    ax.set_xlabel('Percentage of non-street-facing shops (%)')
    ax.set_ylabel('Percentage of non-ground-floor shops (%)')
    ax.set_title(f'Changes for cities (Tier {tier})')
    # 
    output_path = f'/Volumes/IMAC/Research/Figures/Sub-figures/cityplotP_Tier{tier}.eps'
    plt.savefig(output_path, dpi=300, bbox_inches='tight')
    # 
    plt.show()

In [ ]:
# 
results = []
# 
for city in Data['City'].unique():
    #
    city_data = Data[Data['City'] == city]
    # 
    data_2015 = city_data[city_data['Year'] == 2015]
    data_2019 = city_data[city_data['Year'] == 2019]
    data_2023 = city_data[city_data['Year'] == 2023]
    # 
    if not (len(data_2015) == len(data_2019) == len(data_2023) == 1): continue
    # 
    p_horizontal_2015 = data_2015.iloc[0]['P_Horizontal'] * 100
    p_vertical_2015 = data_2015.iloc[0]['P_Vertical'] * 100
    p_horizontal_2019 = data_2019.iloc[0]['P_Horizontal'] * 100
    p_vertical_2019 = data_2019.iloc[0]['P_Vertical'] * 100
    p_horizontal_2023 = data_2023.iloc[0]['P_Horizontal'] * 100
    p_vertical_2023 = data_2023.iloc[0]['P_Vertical'] * 100
    # 
    change_2015_2019_h = p_horizontal_2019 - p_horizontal_2015
    change_2015_2019_v = p_vertical_2019 - p_vertical_2015
    change_2019_2023_h = p_horizontal_2023 - p_horizontal_2019
    change_2019_2023_v = p_vertical_2023 - p_vertical_2019
    # 
    def classify_changes(h_change, v_change):
        #
        if h_change < 0 and v_change < 0: return 'N_HV'
        elif h_change > 0 and v_change < 0: return 'HNV'
        elif h_change < 0 and v_change > 0: return 'NHV'
        elif h_change > 0 and v_change > 0: return 'HV'
    #
    category_2015_2019 = classify_changes(change_2015_2019_h, change_2015_2019_v)
    category_2019_2023 = classify_changes(change_2019_2023_h, change_2019_2023_v)
    # 
    results.append({'CityName': data_2015.iloc[0]['CityName_x'],'TimePeriod': '2015-2019','P_Horizontal_Change': change_2015_2019_h,
        'P_Vertical_Change': change_2015_2019_v,'Category': category_2015_2019})
    #
    results.append({'CityName': data_2015.iloc[0]['CityName_x'],'TimePeriod': '2019-2023','P_Horizontal_Change': change_2019_2023_h,
        'P_Vertical_Change': change_2019_2023_v,'Category': category_2019_2023})
# 转为 DataFrame
results_df = pd.DataFrame(results)
results_df.to_csv('/Volumes/IMAC/Research/Figures/Sub-figures/changes.csv')

# Sankey Figure #

In [ ]:
#
results_df = pd.read_csv('/Volumes/IMAC/Research/全国点评研究/Figures/Sub-figures/changes.csv')
results_df['id'] = results_df['Unnamed: 0'].apply(lambda x: int(x/2))
data_2015_2019 = results_df[results_df['TimePeriod'] == '2015-2019']
data_2019_2023 = results_df[results_df['TimePeriod'] == '2019-2023']
# 
merged_data = pd.merge(data_2015_2019[['CityName', 'Category','id']], data_2019_2023[['CityName', 'Category','id']], on='id', 
                       suffixes=('_2015_2019', '_2019_2023'))
#
grouped_data = merged_data.groupby(['Category_2015_2019', 'Category_2019_2023']).size().reset_index(name='count')
grouped_data['ratio'] = grouped_data['count']/287

In [ ]:
# 
source_categories = grouped_data['Category_2015_2019'].unique()
target_categories = grouped_data['Category_2019_2023'].unique()
# 
links = []
#
for source_category in source_categories:
    #
    for target_category in target_categories:
        #
        value = grouped_data[(grouped_data['Category_2015_2019'] == source_category) & 
        (grouped_data['Category_2019_2023'] == target_category)]['count'].values
        # 
        if len(value) > 0 and value[0] > 0:
            #
            links.append({'source': source_category, 'target': target_category, 'value': value[0]})
# 
links_df = pd.DataFrame(links)
# 
links_df['source'] = '2015-2019' + links_df['source'].astype(str)
links_df['target'] = '2019-2023' + links_df['target'].astype(str)
# 
all_nodes = list(links_df['source'].unique()) + list(links_df['target'].unique())
node_indices = {node: idx for idx, node in enumerate(all_nodes)}
# 
links_df['source'] = links_df['source'].map(node_indices)
links_df['target'] = links_df['target'].map(node_indices)
# 
node_colors = ['#528fac', '#6eaed1', '#fed06e', '#dc6e57'] * (len(all_nodes) // 4 + 1)  # Repeat colors to match the number of nodes
node_colors = node_colors[:len(all_nodes)]  # Trim to match the number of nodes
# 
node_colors_dict = {node: node_colors[idx] for idx, node in enumerate(all_nodes)}
# 
fig = go.Figure(go.Sankey(node=dict(pad=15,thickness=10, widthline=dict(color='black', width=0.5),color=[node_colors_dict[node] for node in all_nodes]),
                          link=dict(source=links_df['source'],target=links_df['target'],value=links_df['value'],
                                    color=[node_colors_dict[all_nodes[idx]] for idx in links_df['source']], hovertemplate="Value: %{value}",
                                    hoverinfo="all" )))
# 
fig.update_layout(title_text='Flow between Categories: 2015-2019 to 2019-2023',font_size=15,width=1000,height=800)
fig.show()

In [ ]:
#
source_categories = grouped_data['Category_2015_2019'].unique()
target_categories = grouped_data['Category_2019_2023'].unique()
# 
links = []
#
for source_category in source_categories:
    #
    for target_category in target_categories:
        #
        value = grouped_data[(grouped_data['Category_2015_2019'] == source_category) & 
        (grouped_data['Category_2019_2023'] == target_category)]['count'].values
        # 
        if len(value) > 0 and value[0] > 0:links.append({'source': source_category, 'target': target_category, 'value': value[0]})
# 
links_df = pd.DataFrame(links)
#
links_df['source'] = '2015-2019' + links_df['source'].astype(str)
links_df['target'] = '2019-2023' + links_df['target'].astype(str)
# 
all_nodes = list(links_df['source'].unique()) + list(links_df['target'].unique())
node_indices = {node: idx for idx, node in enumerate(all_nodes)}
# 
links_df['source'] = links_df['source'].map(node_indices)
links_df['target'] = links_df['target'].map(node_indices)
#
node_colors = ['#528fac', '#6eaed1', '#fed06e', '#dc6e57'] * (len(all_nodes) // 4 + 1)  # Repeat colors to match the number of nodes
node_colors = node_colors[:len(all_nodes)]  # Trim to match the number of nodes
# 
node_colors_dict = {node: node_colors[idx] for idx, node in enumerate(all_nodes)}
# 
fig = go.Figure(go.Sankey(node=dict(pad=15,thickness=10,line=dict(color='black', width=0.5),color=[node_colors_dict[node] for node in all_nodes]),
                          link=dict(source=links_df['source'],target=links_df['target'],value=links_df['value'],
                                    color=[node_colors_dict[all_nodes[idx]] for idx in links_df['source']],hovertemplate="Value: %{value}",
                                    hoverinfo="all")))
# 
fig.update_layout(title_text='Flow between Categories: 2015-2019 to 2019-2023',font_size=15,width=1000,height=800)
fig.show()

In [ ]:
#
cities = Data['City'].unique()
colors = {2015: '#6eaed1', 2019:'#fed06e', 2023:'#dc6e57'}
# 
tier_styles = {'Tier 1': {'marker': 'o', 'color': '#6eaed1'},'Tier 2': {'marker': 's', 'color': '#6eaed1'},'Tier 3': {'marker':'^', 'color': '#6eaed1'},
               'Tier 4': {'marker': 'D', 'color': '#6eaed1'},'Tier 5': {'marker': 'x', 'color': '#6eaed1'}}
legend_labels = {tier: patches.Patch(color=style['color'], label=tier) for tier, style in tier_styles.items()}
# 
for tier, style in tier_styles.items():
    # 
    fig, ax = plt.subplots(figsize=(4, 4), dpi=300)
    # 
    cities_in_tier = Data[Data['CityTier'] == tier]['City'].unique()
    # 
    for city in cities_in_tier:
        #
        city_data = Data[Data['City'] == city]
        city_years = city_data['Year'].unique()
        city_data['P_Horizontal'] = city_data['P_Horizontal'] * 100
        city_data['P_Vertical'] = city_data['P_Vertical'] * 100
        # 
        if set([2015, 2019, 2023]).issubset(city_years):
            # 
            coords_2015 = city_data[city_data['Year'] == 2015].iloc[0]
            coords_2019 = city_data[city_data['Year'] == 2019].iloc[0]
            coords_2023 = city_data[city_data['Year'] == 2023].iloc[0]
            city_name = coords_2015['CityName_x']  # 使用 CityName_x 字段
            # 
            dx_1 = coords_2019['P_Horizontal'] - coords_2015['P_Horizontal']
            dy_1 = coords_2019['P_Vertical'] - coords_2015['P_Vertical']
            dx_2 = coords_2023['P_Horizontal'] - coords_2019['P_Horizontal']
            dy_2 = coords_2023['P_Vertical'] - coords_2019['P_Vertical']
            # 
            arrow1 = patches.FancyArrowPatch((coords_2015['P_Horizontal'], coords_2015['P_Vertical']),
                                             (coords_2015['P_Horizontal'] + dx_1, coords_2015['P_Vertical'] + dy_1),
                                             color='#fde3d6', arrowstyle='->', lw=1.5)
            #
            ax.add_patch(arrow1)
            # 
            arrow2 = patches.FancyArrowPatch((coords_2019['P_Horizontal'], coords_2019['P_Vertical']),
                                             (coords_2019['P_Horizontal'] + dx_2, coords_2019['P_Vertical'] + dy_2),
                                             color='#fed06e', arrowstyle='->', lw=1.5)
            #
            ax.add_patch(arrow2)
            # 
            ax.plot(coords_2015['P_Horizontal'], coords_2015['P_Vertical'], marker=style['marker'], color=colors[2015], label=f'{city_name} 2015')
            ax.plot(coords_2019['P_Horizontal'], coords_2019['P_Vertical'], marker=style['marker'], color=colors[2019], label=f'{city_name} 2019')
            ax.plot(coords_2023['P_Horizontal'], coords_2023['P_Vertical'], marker=style['marker'], color=colors[2023], label=f'{city_name} 2023')
            # 
            ax.text(coords_2015['P_Horizontal'], coords_2015['P_Vertical'], city_name, fontsize=4, ha='right')
            ax.text(coords_2019['P_Horizontal'], coords_2019['P_Vertical'], city_name, fontsize=4, ha='right')
            ax.text(coords_2023['P_Horizontal'], coords_2023['P_Vertical'], city_name, fontsize=4, ha='right')
    # 
    ax.set_xlabel('Percentage of non-street-facing shops (%)')
    ax.set_ylabel('Percentage of non-ground-floor shops (%)')
    ax.set_title(f'Changes for cities (Tier {tier})')
    # 
    output_path = f'/Volumes/IMAC/Research/Figures/Sub-figures/cityplotP_Tier{tier}A.eps'
    plt.savefig(output_path, dpi=300, bbox_inches='tight')
    # 
    plt.show() 

# Category Analysis #

In [ ]:
#
Typedata = pd.read_csv('/Volumes/IMAC/Data/297Dianping/Final Analysis/DP15-23/Mall2015-2023TypejoincitySM.csv')
translation = {'餐饮': 'F&B','购物': 'Retail goods','生活服务': 'Daily service','运动健身': 'Sports','休闲娱乐': 'Entertainment','丽人':'Beauty',
    '教育培训':'Training'}
# 
Typedata['Big_cate_New'] = Typedata['Big_cate_New'].replace(translation)
Typedata['P_Horizontal'] = 1.0-Typedata['P_FS20']
Typedata['P_Vertical'] = 1.0-Typedata['P_GF'] 
Typedata.to_csv('/Volumes/IMAC/Research/Figures/Sub-figures/Mall2015-2023Typejoincity287.csv')
# 
colors = {2015: '#6eaed1', 2019:'#fed06e', 2023:'#dc6e57'}
# 
order = ['F&B', 'Retail goods', 'Daily service', 'Sports', 'Entertainment', 'Beauty', 'Training']
columns_to_plot = ['P_Horizontal', 'P_Vertical']
# 
for column in columns_to_plot:  
    # 
    if 'Big_cate_New' not in Typedata.columns: continue  
    # 
    fig, ax = plt.subplots(figsize=(8, 5), dpi=300)
    filtered_data = Typedata.copy() 
    filtered_data[column] = filtered_data[column] * 100  
    # 
    filtered_data['Big_cate_New'] = filtered_data['Big_cate_New'].astype('category')
    # 
    sns.boxplot(data=filtered_data, x='Big_cate_New', y=column, hue='Year', palette=colors, showfliers=False, order=order)
    # S
    plt.xlabel('Type')
    #
    if column == 'P_Horizontal': plt.ylabel('Percentage of non-street-facing shops (%)')
    elif column == 'P_Vertical': plt.ylabel('Percentage of non-ground-floor shops (%)')
    # 
    plt.ylim(0, 100)
    # 
    plt.legend(title='Year', loc='upper left')
    # 
    plt.savefig(f'/Volumes/IMAC/Research/Figures/Sub-figures/boxplot_{column}.eps', dpi=300, bbox_inches='tight')  
    # 
    plt.show()

In [ ]:
#
SmallCatedata = pd.read_csv('/Volumes/IMAC/Data/297Dianping/Final Analysis/DP15-23/Mall2015-2023Small_catejoincitySM.csv')
SmallCatedata['GF'] = SmallCatedata['Count']*SmallCatedata['P_GF'] 
SmallCatedata['SF'] = SmallCatedata['Count']*SmallCatedata['P_FS20'] 
#
groupedSC = SmallCatedata.groupby(['Small_cate_New', 'Year'])[['Count', 'GF', 'SF']].sum().reset_index()
groupedSC['P_GF'] = groupedSC['GF'] / groupedSC['Count'] 
groupedSC['P_SF'] = groupedSC['SF'] / groupedSC['Count']
groupedSC['P_Vertical'] = 1.0-groupedSC['P_GF'] 
groupedSC['P_Horizontal'] = 1.0-groupedSC['P_SF'] 
groupedSC['logPHCount'] = np.log(groupedSC['Count'] - groupedSC['SF']+1)
groupedSC['logPVCount'] = np.log(groupedSC['Count'] - groupedSC['GF']+1)
groupedSC = groupedSC[~(groupedSC['Small_cate_New']=='0')]
Reclass = pd.read_csv('/Volumes/IMAC/Data/297Dianping/Final Analysis/DP15-23/ReclassifySmallcate.csv')
groupedSC = pd.merge(groupedSC,Reclass,on='Small_cate_New')
groupedSC['Cate_New'] = groupedSC['Big_cate_New'] + '-' + groupedSC['Small_cate_New']
groupedSC.to_csv(f'/Volumes/IMAC/Research/Figures/Sub-figures/Smallcateclusterdetailed2.csv')
# 
plt.figure(figsize=(12, 8))
# 
colors = {2015: '#6eaed1', 2019:'#fed06e', 2023:'#dc6e57'}
# 
markers = {'Catering': 'o', 'Retail': '<', 'Sports': 'X', 'Entertainment': '*', 'Beauty': 'D', 'Training': 'P','Life service': '^'}
# 
filtered_groupedSC = groupedSC[groupedSC['Count'] > 100]
filtered_groupedSC['P_Vertical'] = filtered_groupedSC['P_Vertical'] * 100  
filtered_groupedSC['P_Horizontal'] = filtered_groupedSC['P_Horizontal'] * 100 
# 
cmap_custom = ListedColormap([colors[2015], colors[2019], colors[2023]])
# 
g = sns.jointplot(y='P_Vertical', x='P_Horizontal', data=filtered_groupedSC, hue='Year', palette=colors, height=7)
# 
for cat_en, data in filtered_groupedSC.groupby('Category_En'):
    #
    marker = markers.get(cat_en, '>')  
    plt.scatter(data['P_Horizontal'], data['P_Vertical'], label=cat_en, marker=marker, s=50, c=data['Year'], cmap=cmap_custom)
# 
g.ax_joint.legend(loc='upper left')
# 
plt.ylim(0, 62)
plt.xlim(0, 82)
plt.xlabel('Percentage of non-street-facing shops (%)')
plt.ylabel('Percentage of non-ground-floor shops (%)')
plt.savefig(f'/Volumes/IMAC/Research/Figures/Sub-figures/smallcateP.eps', dpi=300, bbox_inches='tight')
# 
plt.show()

In [ ]:
# 
filtered_groupedSC['Total'] = filtered_groupedSC['P_Horizontal'] + filtered_groupedSC['P_Vertical']
# 
top_10_per_year = {}
bottom_10_per_year = {}
#
for year, group in filtered_groupedSC.groupby('Year'):
    #
    top_10_per_year[year] = group.nlargest(10, 'Total')  
    bottom_10_per_year[year] = group.nsmallest(10, 'Total')  
# 
for year in sorted(filtered_groupedSC['Year'].unique()):
    # 
    top_10 = top_10_per_year[year]
    bottom_10 = bottom_10_per_year[year]
    # 
    ellipsis_row = pd.DataFrame({'Type_En': ['...'], 'P_Horizontal': [0],'P_Vertical': [0],'Total': [0]})
    # 
    top_10 = top_10.sort_values(by='Total', ascending=True)
    # 
    bottom_10 = bottom_10.sort_values(by='Total', ascending=True)
    # 
    combined = pd.concat([bottom_10, ellipsis_row, top_10]).reset_index(drop=True)
    # 
    combined['P_Horizontal'] = -combined['P_Horizontal']
    # 
    fig, ax = plt.subplots(figsize=(6, 10))
    # 
    ax.barh(combined['Type_En'], combined['P_Horizontal'], color='#6eaed1', label='P_Horizontal')
    ax.barh(combined['Type_En'], combined['P_Vertical'], left=0, color='#dc6e57', label='P_Vertical')
    # 
    ax.axvline(0, color='black', linewidth=0.8, linestyle='--')
    # 
    ax.legend(loc='upper left', bbox_to_anchor=(1.05, 1), title='Proportion Type')
    # 
    ax.set_title(f'Top and Bottom 10 Categories in {year}', fontsize=14)
    ax.set_xlabel('Proportion (%)')
    ax.set_ylabel('Category')
    # 
    plt.tight_layout()
    plt.savefig(f'/Volumes/IMAC/Research/Figures/Sub-figures/centered_stacked_{year}.eps', dpi=300, bbox_inches='tight')
    plt.show()

In [ ]:
# 
plt.figure(figsize=(5, 4))
#
colors = {2015: '#6eaed1', 2019:'#fed06e', 2023:'#dc6e57'}
# 
markers = {'Catering': 'o', 'Retail': '<', 'Sports': 'X', 'Entertainment': '*', 'Beauty': 'D', 'Training': 'P','Life service': '^'}
# 
filtered_groupedSC = groupedSC[groupedSC['Count'] > 100]
filtered_groupedSC['P_Vertical'] = filtered_groupedSC['P_Vertical'] * 100  # Convert to percentage
filtered_groupedSC['P_Horizontal'] = filtered_groupedSC['P_Horizontal'] * 100  # Convert to percentage
# 
for cat_en, data in filtered_groupedSC.groupby('Category_En'):
    #
    plt.scatter(data['P_Horizontal'],data['P_Vertical'],label=cat_en,marker=markers.get(cat_en, 'o'),s=50,c=data['Year'].map(colors),)
# 
plt.ylim(30, 40)
plt.xlim(40, 60)
plt.xlabel('Percentage of non-street-facing shops (%)')
plt.ylabel('Percentage of non-ground-floor shops (%)')
# 
category_handles = [mlines.Line2D([], [], color='black', marker=marker, linestyle='None', markersize=8, label=cat_en)
                    for cat_en, marker in markers.items()]
#
category_legend = plt.legend(handles=category_handles,title="Category",loc='center left',bbox_to_anchor=(1.05, 0.5),frameon=False)
# 
year_handles = [mlines.Line2D([], [], color=color, marker='o', linestyle='None', markersize=8, label=str(year))for year, color in colors.items()]
# 
plt.gca().add_artist(category_legend)
# 
plt.savefig('/Volumes/IMAC/Research/Figures/Sub-figures/smallcatePS1.eps', dpi=300, bbox_inches='tight')
# 
plt.show()

In [ ]:
# 
plt.figure(figsize=(5, 4))
# 
colors = {2015: '#6eaed1', 2019:'#fed06e', 2023:'#dc6e57'}
# 
markers = {'Catering': 'o', 'Retail': '<', 'Sports': 'X', 'Entertainment': '*', 'Beauty': 'D', 'Training': 'P','Life service': '^'}
# 
filtered_groupedSC = groupedSC[groupedSC['Count'] > 100]
filtered_groupedSC['P_Vertical'] = filtered_groupedSC['P_Vertical'] * 100  
filtered_groupedSC['P_Horizontal'] = filtered_groupedSC['P_Horizontal'] * 100  
# 
for cat_en, data in filtered_groupedSC.groupby('Category_En'):
    #
    plt.scatter(data['P_Horizontal'],data['P_Vertical'],label=cat_en,marker=markers.get(cat_en, 'o'),s=50,c=data['Year'].map(colors),)
# 
plt.ylim(1, 6)
plt.xlim(30, 35)
plt.xlabel('Percentage of non-street-facing shops (%)')
plt.ylabel('Percentage of non-ground-floor shops (%)')
# 
category_handles = [mlines.Line2D([], [], color='black', marker=marker, linestyle='None', markersize=8, label=cat_en)
                    for cat_en, marker in markers.items()]
#
category_legend = plt.legend(handles=category_handles,title="Category",loc='center left',bbox_to_anchor=(1.05, 0.5),frameon=False)
# 
year_handles = [mlines.Line2D([], [], color=color, marker='o', linestyle='None', markersize=8, label=str(year))
                for year, color in colors.items()]
# 
plt.gca().add_artist(category_legend)
# 
plt.savefig('/Volumes/IMAC/Research/Figures/Sub-figures/smallcatePS2.eps', dpi=300, bbox_inches='tight')
# 
plt.show()

# Regreesion Analysis #

In [ ]:
#
Data = pd.read_csv('/Volumes/IMAC/Data/297Dianping/Final Analysis/DP15-23/Mall2015-2023joincitySM.csv')
Data['logPPop'] = np.log(Data['Permanent resident population (10000)']+1)
Data['logGRPP'] = np.log(Data['GRPP (yuan)']+1)
Data['logGRP'] = np.log(Data['GRP (10000 yuan)']+1)
Data['P_Horizontal'] = 1-Data['P_FS20']
Data['P_Vertical'] = 1-Data['P_GF']
#
Data = pd.get_dummies(Data, columns=['Year','CityTier', 'BldClimateZone'])
Data['Year_2015'] = Data['Year_2015'].astype(int)
Data['Year_2019'] = Data['Year_2019'].astype(int)
Data['Year_2023'] = Data['Year_2023'].astype(int)
Data['CityTier_Tier 1'] = Data['CityTier_Tier 1'].astype(int)
Data['CityTier_Tier 2'] = Data['CityTier_Tier 2'].astype(int)
Data['CityTier_Tier 3'] = Data['CityTier_Tier 3'].astype(int)
Data['CityTier_Tier 4'] = Data['CityTier_Tier 4'].astype(int)
Data['CityTier_Tier 5'] = Data['CityTier_Tier 5'].astype(int)
Data['BldClimateZone_I'] = Data['BldClimateZone_I'].astype(int)
Data['BldClimateZone_II'] = Data['BldClimateZone_II'].astype(int)
Data['BldClimateZone_III'] = Data['BldClimateZone_III'].astype(int)
Data['BldClimateZone_IV'] = Data['BldClimateZone_IV'].astype(int)
Data['BldClimateZone_V'] = Data['BldClimateZone_V'].astype(int)
Data['BldClimateZone_VI'] = Data['BldClimateZone_VI'].astype(int)
Data['BldClimateZone_VII'] = Data['BldClimateZone_VII'].astype(int)
#
Policy = pd.read_csv('C:/Users/zej/Desktop/Analysis/Policy.csv')
Data = Data.merge(Policy,how='left',left_on='CityAID',right_on='市代码')
#
Data15 = Data[Data['Year_2015']==1]
Data15['Policy'] = Data15['开墙打洞（破墙开店）整治'].apply(lambda x: 0 if x=='无' else (0 if int(x)>2015 else 1))
#
Data19 = Data[Data['Year_2019']==1]
Data19['Policy'] = Data19['开墙打洞（破墙开店）整治'].apply(lambda x: 0 if x=='无' else (0 if int(x)>2019 else 1))
#
Data23 = Data[Data['Year_2023']==1]
Data23['Policy'] = Data23['开墙打洞（破墙开店）整治'].apply(lambda x: 0 if x=='无' else 1)
#
Data = pd.concat([Data15,Data19,Data23],axis=0)

In [ ]:
# 
Independent = ['logPPop', 'logGRPP', 'Third industry GRP percentage (%)','Year_2019', 'Year_2023', 'BldClimateZone_I',
               'BldClimateZone_II','BldClimateZone_IV', 'BldClimateZone_V', 'BldClimateZone_VI', 'BldClimateZone_VII']
#
Dependent = ['P_Horizontal', 'P_Vertical']
# 
results_summary = {}
#
for col_name in Dependent:
    # 
    y = Data[col_name]
    X = Data[Independent] 
    # 
    X_with_const = sm.add_constant(X)  
    # 
    model = sm.OLS(y, X_with_const)
    result = model.fit()
    # 
    vif_data = pd.DataFrame()
    vif_data["Variable"] = ['const'] + Independent  
    vif_data["VIF"] = [variance_inflation_factor(X_with_const.values, i) for i in range(X_with_const.shape[1])]
    # 
    scaler_X = StandardScaler()
    X_scaled = scaler_X.fit_transform(X)
    # 
    X_scaled_with_const = sm.add_constant(X_scaled)
    # 
    model_scaled = sm.OLS(y, X_scaled_with_const)
    result_scaled = model_scaled.fit() 
    # 
    standardized_coefficients = result_scaled.params
    # 
    results_summary[col_name] = {"model_summary": result.summary().as_text(),"non_standardized_coefficients": result.params,
                                 "standardized_coefficients": standardized_coefficients,"VIF": vif_data}  
    # 
    print(f"=== Regression results for {col_name} (Non-standardized) ===")
    print(result.summary()) 
    # 
    print(f"=== Standardized Coefficients for {col_name} ===")
    print(standardized_coefficients) 
    # 
    print(f"=== VIF for {col_name} ===")
    print(vif_data)  

In [ ]:
#
Independent = ['logPPop', 'logGRPP', 'Third industry GRP percentage (%)','Policy','Year_2019', 'Year_2023', 'BldClimateZone_I',
               'BldClimateZone_II','BldClimateZone_IV', 'BldClimateZone_V', 'BldClimateZone_VI', 'BldClimateZone_VII']
Dependent = ['P_Horizontal', 'P_Vertical']
#
results_summary = {}
#
for col_name in Dependent:
    # 
    y = Data[col_name]
    X = Data[Independent]
    # 
    X_with_const = sm.add_constant(X)
    # 
    model = sm.OLS(y, X_with_const)
    result = model.fit()
    # 
    vif_data = pd.DataFrame()
    vif_data["Variable"] = ['const'] + Independent  
    vif_data["VIF"] = [variance_inflation_factor(X_with_const.values, i) for i in range(X_with_const.shape[1])]
    # 
    scaler_X = StandardScaler()
    X_scaled = scaler_X.fit_transform(X)
    # 
    X_scaled_with_const = sm.add_constant(X_scaled)
    # 
    model_scaled = sm.OLS(y, X_scaled_with_const)
    result_scaled = model_scaled.fit()
    # 
    standardized_coefficients = result_scaled.params
    # 
    results_summary[col_name] = {"model_summary": result.summary().as_text(),"non_standardized_coefficients": result.params,
                                 "standardized_coefficients": standardized_coefficients,"VIF": vif_data}
    # 
    print(f"=== Regression results for {col_name} (Non-standardized) ===")
    print(result.summary())  
    # 
    print(f"=== Standardized Coefficients for {col_name} ===")
    print(standardized_coefficients)  
    # 
    print(f"=== VIF for {col_name} ===")
    print(vif_data)  

In [ ]:
# 
corr = X.corr()
# 
mask = np.triu(np.ones_like(corr, dtype=bool))
# 
f, ax = plt.subplots(figsize=(10, 10))
# 
sns.set(style="white") 
# 
sns.heatmap(corr, mask=mask, cmap='RdBu_r', annot=True, fmt='.2f',annot_kws={"fontsize": 12, "fontweight": "bold", "fontstyle": "italic"},
            square=True, linewidths=.5, cbar_kws={"shrink": .5},vmin=-1, vmax=1,cbar=True, xticklabels=True, yticklabels=True)
# 
plt.show()

In [ ]:
# 
for col in ['logPPop', 'logGRPP','Third industry GRP percentage (%)']:
    #
    plt.scatter(Data[col], Data['P_Horizontal'])
    plt.title(f'Scatter plot: {col} vs Y')
    plt.xlabel(col)
    plt.ylabel('Y')
    plt.show()

In [ ]:
# 
for col in ['logPPop', 'logGRPP','Third industry GRP percentage (%)']:
    #
    plt.scatter(Data[col], Data['P_Vertical'])
    plt.title(f'Scatter plot: {col} vs Y')
    plt.xlabel(col)
    plt.ylabel('Y')
    plt.show()

In [ ]:
# 
sm.qqplot(residuals, line='45')
plt.title('Q-Q Plot of Residuals')
plt.show()
# Shapiro-Wilk正态性检验
stat, p = shapiro(residuals)
print(f'Shapiro-Wilk test: stat={stat:.3f}, p-value={p:.3f}')
#
if p > 0.05: print("Residuals appear to be normally distributed.")
else: print("Residuals do not appear to be normally distributed.")

In [ ]:
# 
bp_test = het_breuschpagan(residuals, model.model.exog)
labels = ['LM Statistic', 'p-value', 'F-Statistic', 'F p-value']
print(dict(zip(labels, bp_test)))
#
if bp_test[1] > 0.05: print("Residuals are homoscedastic (constant variance).")
else: print("Residuals are heteroscedastic (non-constant variance).")

In [ ]:
#
Independent = ['logPPop', 'logGRPP','Third industry GRP percentage (%)','Year_2019', 'Year_2023','BldClimateZone_I', 'BldClimateZone_II', 
               'BldClimateZone_IV', 'BldClimateZone_V', 'BldClimateZone_VI', 'BldClimateZone_VII']
Dependent = ['P_Horizontal', 'P_Vertical']
# 
all_columns = Independent + Dependent
# 
data_subset = Data[all_columns]
# 
description = data_subset.describe()

In [ ]:
#
Data = Data[Data['Policy']==0]
# 
Independent = ['logPPop', 'logGRPP', 'Third industry GRP percentage (%)','Year_2019', 'Year_2023', 'BldClimateZone_I', 'BldClimateZone_II',
               'BldClimateZone_IV', 'BldClimateZone_V', 'BldClimateZone_VI', 'BldClimateZone_VII','Policy']
Dependent = ['P_Horizontal', 'P_Vertical']
# 
results_summary = {}
#
for col_name in Dependent:
    # 
    y = Data[col_name]
    X = Data[Independent]
    # 
    X_with_const = sm.add_constant(X)
    # 
    model = sm.OLS(y, X_with_const)
    result = model.fit()
    # 
    vif_data = pd.DataFrame()
    vif_data["Variable"] = ['const'] + Independent 
    vif_data["VIF"] = [variance_inflation_factor(X_with_const.values, i) for i in range(X_with_const.shape[1])]
    # 
    scaler_X = StandardScaler()
    X_scaled = scaler_X.fit_transform(X)
    # 
    X_scaled_with_const = sm.add_constant(X_scaled)
    # 
    model_scaled = sm.OLS(y, X_scaled_with_const)
    result_scaled = model_scaled.fit()
    # 
    standardized_coefficients = result_scaled.params
    # 
    results_summary[col_name] = {"model_summary": result.summary().as_text(),"non_standardized_coefficients": result.params,
                                 "standardized_coefficients": standardized_coefficients,"VIF": vif_data}
    # 
    print(f"=== Regression results for {col_name} (Non-standardized) ===")
    print(result.summary())  
    # 
    print(f"=== Standardized Coefficients for {col_name} ===")
    print(standardized_coefficients)  
    # 
    print(f"=== VIF for {col_name} ===")
    print(vif_data) 

# Robustness Test - 1 #

In [ ]:
#
DataRT = pd.read_csv('/Volumes/IMAC/Data/297Dianping/Final Analysis/DP15-23/Mall2015-2023joincityRT.csv')
DataRT['P_Horizontal'] = 1.0-DataRT['P_RTFS20']
DataRT['P_Vertical'] = 1.0-DataRT['P_GF'] 
DataRT = DataRT[DataRT['P_RTFS20']>0]
DataRT = DataRT.groupby('City').filter(lambda x: x['Year'].nunique() == 3)
DataRT = DataRT[~(DataRT['CityTier']=='Tier 5')]
# 
DataRT['NSF'] = DataRT['Count'] * DataRT['P_Horizontal']
DataRT['NGF'] = DataRT['Count'] * DataRT['P_Vertical']
#
Data = pd.read_csv('/Volumes/IMAC/Data/297Dianping/Final Analysis/DP15-23/Mall2015-2023joincitySM.csv')
Data['P_Horizontal'] = 1.0-Data['P_FS20']
Data['P_Vertical'] = 1.0-Data['P_GF'] 
Data = Data[Data['City'].isin(DataRT['City'])]
#
Data['NSF'] = Data['Count'] * Data['P_Horizontal']
Data['NGF'] = Data['Count'] * Data['P_Vertical']

## Robustness Test - 1.1 ##

In [ ]:
# 
result = Data.groupby('Year', as_index=False)[['Count', 'NGF', 'NSF']].sum()
result['P_NGF'] = result['NGF'] / result['Count'] * 100  # Convert to percentage
result['P_NSF'] = result['NSF'] / result['Count'] * 100  # Convert to percentage
result['Count'] = result['Count'] / 1_000_000          # Convert to millions
# 
colors = {2015: '#6eaed1', 2019: '#fed06e', 2023: '#dc6e57'}
#
columns = ['Count', 'P_NGF', 'P_NSF']
y_labels = ['Total count (millions)', 'Percentage of non-street-facing shops (%)']
# D
dual_axis_columns = [('NSF', 'P_NSF', 'Count of non-street-facing shops (millions)', 'Percentage of non-street-facing shops')]
#
for left_col, right_col, left_label, right_label in dual_axis_columns:
    plt.figure(figsize=(4, 4)) 
    #
    bar_width = 0.9  # Bar width
    bars = plt.bar(result['Year'], result[left_col] / 1_000_000, color=[colors[year] for year in result['Year']], width=bar_width)
    # 
    for bar in bars:
        height = bar.get_height()
        plt.text(bar.get_x() + bar.get_width() / 2, height, f'{height:.2f}', ha='center', va='bottom', fontsize=10)
    # 
    plt.xlabel('Year', fontsize=12)
    plt.ylabel(left_label, fontsize=10)
    plt.xticks(result['Year'], fontsize=10)
    plt.ylim(0, 10)  
    plt.yticks(fontsize=10)
    # 
    ax2 = plt.gca().twinx()  
    ax2.plot(result['Year'], result[right_col], color='red', marker='o', label=right_label)
    ax2.set_ylabel(right_label, fontsize=10, color='red')
    ax2.tick_params(axis='y', labelcolor='red', labelsize=10)
    ax2.set_ylim(0, 52)  
    # 
    offset = 50 * 0.03  
    for x, y in zip(result['Year'], result[right_col]):
        #
        ax2.text(x, y - offset, f'{y:.2f}%', ha='center', va='top', fontsize=10, color='red') 
    # 
    plt.tight_layout()
    filename = f"/Volumes/IMAC/Research/Figures/Sub-figures/RT1{right_col}_by_Year.eps"
    plt.savefig(filename, format="eps")
    plt.show()

In [ ]:
# 
result = DataRT.groupby('Year', as_index=False)[['Count', 'NGF', 'NSF']].sum()
result['P_NGF'] = result['NGF'] / result['Count'] * 100  
result['P_NSF'] = result['NSF'] / result['Count'] * 100  
result['Count'] = result['Count'] / 1_000_000         
# 
colors = {2015: '#6eaed1', 2019: '#fed06e', 2023: '#dc6e57'}
# 
columns = ['Count',  'P_NSF']
y_labels = ['Total count (millions)', 'Percentage of non-street-facing shops (%)']
# 
colors = {2015: '#6eaed1', 2019: '#fed06e', 2023: '#dc6e57'}
# 
dual_axis_columns = [ ('NSF', 'P_NSF', 'Count of non-street-facing shops (millions)', 'Percentage of non-street-facing shops')]
#
for left_col, right_col, left_label, right_label in dual_axis_columns:
    #
    plt.figure(figsize=(4, 4))  
    # 
    bar_width = 0.9  
    bars = plt.bar(result['Year'], result[left_col] / 1_000_000, color=[colors[year] for year in result['Year']], width=bar_width)
    # 
    for bar in bars:
        #
        height = bar.get_height()
        plt.text(bar.get_x() + bar.get_width() / 2, height, f'{height:.2f}', ha='center', va='bottom', fontsize=10)
    # 
    plt.xlabel('Year', fontsize=12)
    plt.ylabel(left_label, fontsize=10)
    plt.xticks(result['Year'], fontsize=10)
    plt.ylim(0, 10) 
    plt.yticks(fontsize=10)
    #
    ax2 = plt.gca().twinx()  
    ax2.plot(result['Year'], result[right_col], color='red', marker='o', label=right_label)
    ax2.set_ylabel(right_label, fontsize=10, color='red')
    ax2.tick_params(axis='y', labelcolor='red', labelsize=10)
    ax2.set_ylim(0, 52) 
    # 
    offset = 50 * 0.03  
    # 
    for x, y in zip(result['Year'], result[right_col]):ax2.text(x, y - offset, f'{y:.2f}%', ha='center', va='top', fontsize=10, color='red') 
    # 
    plt.tight_layout()
    filename = f"/Volumes/IMAC/Research/全国点评研究/Figures/Sub-figures/RT2{right_col}_by_Year.eps"
    plt.savefig(filename, format="eps")
    plt.show()

## Robustness Test - 1.2 ##

In [ ]:
# 
sns.set_theme(style="white")
sns.set_style('white', rc={'xtick.bottom': True,'ytick.left': True,})
# Define the color palette for the plot based on the years
colors = {2015: '#6eaed1', 2019:'#fed06e', 2023:'#dc6e57'}
columns_to_plot = ['P_Horizontal']
#
LP = [0.033, 0.0715]
i = 0
# 
for column in columns_to_plot:
    # 
    filtered_data = Data[(Data[column] != 0) & (Data[column] != 1)]
    # 
    filtered_data = filtered_data.groupby('City').filter(lambda x: len(x) == 3)
    # 
    filtered_data[column] = filtered_data[column] * 100
    # 
    years = filtered_data['Year'].unique()
    # 
    fig, ax = plt.subplots(figsize=(8, 5), dpi=300)
    # 
    sns.kdeplot(data=filtered_data, x=column, hue='Year', fill=True, bw_adjust=0.15, palette=colors, hue_order=sorted(years))
    # 
    for year in sorted(years):
        # 
        mean_value = filtered_data[filtered_data['Year'] == year][column].mean()
        # 
        color = colors[year]
        # 
        plt.axvline(mean_value, color=color, linestyle='dashed', linewidth=1, label=f'Mean ({year})')
        # 
        label_position = LP[i] 
        PositionX = mean_value
        if (year == 2023) & (column == 'P_Vertical'): PositionX = mean_value + 0.8
        if (year == 2019) & (column == 'P_Vertical'): PositionX = mean_value - 0.5
        plt.text(PositionX, label_position, f'{mean_value:.2f}', color=color, ha='center', va='center', fontsize=10)
    #
    i += 1
    #
    if column == 'P_Horizontal': plt.xlabel('Percentage of non-street-facing shops (%)', fontsize=12)
    elif column == 'P_Vertical': plt.xlabel('Percentage of non-ground-floor shops (%)', fontsize=12)
    #
    plt.ylabel('Density', fontsize=12)
    # 
    plt.legend(title='Year', loc='upper right', fontsize=10)
    # 
    plt.tick_params(axis='both', direction='out', length=6, width=1) 
    # 
    ax.minorticks_off()
    # 
    ax.set_ylim(0, 0.052) 
    plt.gca().yaxis.set_major_formatter(FuncFormatter(lambda x, _: f'{x:.2f}')) 
    # 
    plt.savefig(f'/Volumes/IMAC/Research/Figures/Sub-figures/kde_plot_{column}RT1.eps', dpi=300, bbox_inches='tight')
    # 
    plt.show()

In [ ]:
# 
colors = {2015: '#6eaed1', 2019:'#fed06e', 2023:'#dc6e57'}
columns_to_plot = ['P_Horizontal']
#
LP = [0.033, 0.0715]
i = 0
# 
for column in columns_to_plot:
    # 
    filtered_data = DataRT[(DataRT[column] != 0) & (Data[column] != 1)]
    # 
    filtered_data = filtered_data.groupby('City').filter(lambda x: len(x) == 3)
    # 
    filtered_data[column] = filtered_data[column] * 100
    # 
    years = filtered_data['Year'].unique()
    # 
    fig, ax = plt.subplots(figsize=(8, 5), dpi=300)
    # 
    sns.kdeplot(data=filtered_data, x=column, hue='Year', fill=True, bw_adjust=0.15, palette=colors, hue_order=sorted(years))
    # 
    for year in sorted(years):
        # 
        mean_value = filtered_data[filtered_data['Year'] == year][column].mean()
        # 
        color = colors[year]
        # 
        plt.axvline(mean_value, color=color, linestyle='dashed', linewidth=1, label=f'Mean ({year})')
        # 
        label_position = LP[i]  # P
        PositionX = mean_value
        if (year == 2023) & (column == 'P_Vertical'): PositionX = mean_value + 0.8
        if (year == 2019) & (column == 'P_Vertical'): PositionX = mean_value - 0.5
        plt.text(PositionX, label_position, f'{mean_value:.2f}', color=color, ha='center', va='center', fontsize=10)
    #
    i += 1
    # A
    if column == 'P_Horizontal': plt.xlabel('Percentage of non-street-facing shops (%)', fontsize=12)
    elif column == 'P_Vertical': plt.xlabel('Percentage of non-ground-floor shops (%)', fontsize=12)
    #
    plt.ylabel('Density', fontsize=12)
    # 
    plt.legend(title='Year', loc='upper right', fontsize=10)
    # 
    plt.tick_params(axis='both', direction='out', length=6, width=1)  
    # 
    ax.minorticks_off()
    # 
    ax.set_ylim(0, 0.052) 
    plt.gca().yaxis.set_major_formatter(FuncFormatter(lambda x, _: f'{x:.2f}')) 
    # 
    plt.savefig(f'/Volumes/IMAC/Research/全国点评研究/Figures/Sub-figures/kde_plot_{column}RT2.eps', dpi=300, bbox_inches='tight')
    # 
    plt.show()

## Robustness Test - 1.3 ##

In [ ]:
#
colors = {2015: '#6eaed1', 2019:'#fed06e', 2023:'#dc6e57'}
tier_order = ['Tier 1', 'Tier 2', 'Tier 3', 'Tier 4', 'Tier 5']
columns_to_plot = ['P_Horizontal']
sns.set_style('white', rc={'xtick.bottom': True,'ytick.left': True,})
# 
for i, column in enumerate(columns_to_plot):  
    # 
    filtered_data = Data[(Data[column] != 0) & (Data[column] != 1)]
    #
    filtered_data = filtered_data.groupby('City').filter(lambda x: len(x) == 3)
    #
    filtered_data[column] = filtered_data[column] * 100
    # 
    fig, ax = plt.subplots(figsize=(8, 5), dpi=300)
    # 
    years = filtered_data['Year'].unique()
    # D
    sns.violinplot(data=filtered_data, x='CityTier', y=column, hue='Year', inner="quart", fill=False, palette=colors)
    #
    plt.xlabel('City tier')
    if column == 'P_Horizontal': plt.ylabel('Percentage of non-street-facing shops (%)')
    elif column == 'P_Vertical': plt.ylabel('Percentage of non-ground-floor shops (%)')
    # S
    plt.ylim(0, 75)
    # 
    if i == 0: plt.legend(title='Year', loc='lower left')
    else: plt.legend(title='Year', loc='upper right')
    # 
    plt.tick_params(axis='both', direction='out', length=6, width=1)  # Major ticks
    # 
    ax.minorticks_off()
    # 
    plt.savefig(f'/Volumes/IMAC/Research/Figures/Sub-figures/violinplot_{column}RT1.eps', dpi=300, bbox_inches='tight')
    #
    plt.show()

In [ ]:
# 
colors = {2015: '#6eaed1', 2019:'#fed06e', 2023:'#dc6e57'}
tier_order = ['Tier 1', 'Tier 2', 'Tier 3', 'Tier 4', 'Tier 5']
columns_to_plot = ['P_Horizontal']
sns.set_style('white', rc={'xtick.bottom': True,'ytick.left': True,})
#
for i, column in enumerate(columns_to_plot):  
    # 
    filtered_data = DataRT[(DataRT[column] != 0) & (DataRT[column] != 1)]
    #
    filtered_data = filtered_data.groupby('City').filter(lambda x: len(x) == 3)
    # 
    filtered_data[column] = filtered_data[column] * 100
    # 
    fig, ax = plt.subplots(figsize=(8, 5), dpi=300)
    # 
    years = filtered_data['Year'].unique()
    #
    sns.violinplot(data=filtered_data, x='CityTier', y=column, hue='Year', inner="quart", fill=False, palette=colors)
    # 
    plt.xlabel('City tier')
    #
    if column == 'P_Horizontal': plt.ylabel('Percentage of non-street-facing shops (%)')
    elif column == 'P_Vertical': plt.ylabel('Percentage of non-ground-floor shops (%)')
    # 
    plt.ylim(0, 75)
    # 
    if i == 0: plt.legend(title='Year', loc='lower left')
    else: plt.legend(title='Year', loc='upper right')
    # 
    plt.tick_params(axis='both', direction='out', length=6, width=1)  
    # 
    ax.minorticks_off()
    #
    plt.savefig(f'/Volumes/IMAC/Research/Figures/Sub-figures/violinplot_{column}RT2.eps', dpi=300, bbox_inches='tight')
    # 
    plt.show()